In [9]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 11, Finished, Available, Finished, False)

#### <mark>**API Source Ingestion Handling**</mark>
**We will use this API as an external product enrichment source.**

<mark>**Goal:**</mark>
- **Ingest external product data from a public REST API**
- **Land raw JSON into Bronze**
- **Flatten and standardize into Silver**
- **Cover schema handling, pagination concept, and retry concept**

- **1. API request**
- **2. Raw JSON ingestion**
- **3. Schema inspection**
- **4. Schema handling**
- **5. Nested JSON flattening**
- **6. Bronze table**
- **7. Silver table**
- **8. Pagination pattern**
- **9. Retry pattern**

###### <mark>**We are using Public API**</mark>
**<u>https://dummyjson.com/products</u>**

###### <mark>**Project role**</mark>

**It will represent an external marketplace/supplier/reference API that provides:**

- product attributes
- brand
- stock
- ratings
- availability
- review-related information

###### <mark>**Step 1: Import libraries**</mark>

In [10]:
# requests is used to call REST APIs over HTTP
import requests

# Spark functions help us rename/select nested fields cleanly
from pyspark.sql.functions import col, lit

# Optional: json can help if we want to inspect raw payloads later
import json

# time is useful for retry wait/sleep logic
import time

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 12, Finished, Available, Finished, False)

###### <mark>**Step 2: Basic API call**</mark>

In [11]:
# Public product API endpoint
url = "https://dummyjson.com/products"

# Send GET request to API
response = requests.get(url, timeout=30)

# Check HTTP status code
print("Status code:", response.status_code)

# Convert response to Python JSON object
data = response.json()

# View top-level keys
print(data.keys())

# View sample product records
print(data["products"][:2])

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 13, Finished, Available, Finished, False)

Status code: 200
dict_keys(['products', 'total', 'skip', 'limit'])
[{'id': 1, 'title': 'Essence Mascara Lash Princess', 'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.', 'category': 'beauty', 'price': 9.99, 'discountPercentage': 10.48, 'rating': 2.56, 'stock': 99, 'tags': ['beauty', 'mascara'], 'brand': 'Essence', 'sku': 'BEA-ESS-ESS-001', 'weight': 4, 'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99}, 'warrantyInformation': '1 week warranty', 'shippingInformation': 'Ships in 3-5 business days', 'availabilityStatus': 'In Stock', 'reviews': [{'rating': 3, 'comment': 'Would not recommend!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Eleanor Collins', 'reviewerEmail': 'eleanor.collins@x.dummyjson.com'}, {'rating': 4, 'comment': 'Very satisfied!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Lucas Gordon', 'reviewe

###### <mark>**Step 3: Define schema explicitly**</mark>

In [12]:
from pyspark.sql.types import *

schema = StructType([
    StructField("id", LongType(), True),
    StructField("title", StringType(), True),
    StructField("description", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("discountPercentage", DoubleType(), True),
    StructField("rating", DoubleType(), True),
    StructField("stock", LongType(), True),
    StructField("availabilityStatus", StringType(), True),
    StructField("sku", StringType(), True),
    StructField("weight", DoubleType(), True),
    StructField("dimensions", StructType([
        StructField("width", DoubleType(), True),
        StructField("height", DoubleType(), True),
        StructField("depth", DoubleType(), True)
    ]), True),
    StructField("shippingInformation", StringType(), True),
    StructField("returnPolicy", StringType(), True),
    StructField("minimumOrderQuantity", LongType(), True),
    StructField("meta", StructType([
        StructField("createdAt", StringType(), True),
        StructField("updatedAt", StringType(), True),
        StructField("barcode", StringType(), True),
        StructField("qrCode", StringType(), True)
    ]), True),
    StructField("thumbnail", StringType(), True)
])

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 14, Finished, Available, Finished, False)

###### <mark>**Step 3A: Normalize API payload before Spark DataFrame creation**</mark>

In [13]:
# Step 4: Normalize API payload before creating Spark DataFrame
# Why:
# The API contains some numeric fields as Python int values, but our schema expects DoubleType.
# We convert those fields to float explicitly so Spark can accept them.

normalized_products = []

for product in data["products"]:
    product_copy = dict(product)

    # Convert top-level numeric fields to float where schema expects DoubleType
    if product_copy.get("price") is not None:
        product_copy["price"] = float(product_copy["price"])

    if product_copy.get("discountPercentage") is not None:
        product_copy["discountPercentage"] = float(product_copy["discountPercentage"])

    if product_copy.get("rating") is not None:
        product_copy["rating"] = float(product_copy["rating"])

    if product_copy.get("weight") is not None:
        product_copy["weight"] = float(product_copy["weight"])

    # Convert nested dimensions values to float
    if product_copy.get("dimensions") is not None:
        dims = dict(product_copy["dimensions"])

        if dims.get("width") is not None:
            dims["width"] = float(dims["width"])

        if dims.get("height") is not None:
            dims["height"] = float(dims["height"])

        if dims.get("depth") is not None:
            dims["depth"] = float(dims["depth"])

        product_copy["dimensions"] = dims

    normalized_products.append(product_copy)

# Optional quick check
print(normalized_products[:2])

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 15, Finished, Available, Finished, False)

[{'id': 1, 'title': 'Essence Mascara Lash Princess', 'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.', 'category': 'beauty', 'price': 9.99, 'discountPercentage': 10.48, 'rating': 2.56, 'stock': 99, 'tags': ['beauty', 'mascara'], 'brand': 'Essence', 'sku': 'BEA-ESS-ESS-001', 'weight': 4.0, 'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99}, 'warrantyInformation': '1 week warranty', 'shippingInformation': 'Ships in 3-5 business days', 'availabilityStatus': 'In Stock', 'reviews': [{'rating': 3, 'comment': 'Would not recommend!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Eleanor Collins', 'reviewerEmail': 'eleanor.collins@x.dummyjson.com'}, {'rating': 4, 'comment': 'Very satisfied!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Lucas Gordon', 'reviewerEmail': 'lucas.gordon@x.dummyjson.com'}, {'rating': 5, 'comment'

###### <mark>**Step 3B: Convert normalized API response to Spark DataFrame**</mark>

In [14]:
# Step 5: Convert normalized API response to Spark DataFrame
# We now use normalized_products instead of raw data["products"]

api_df = spark.createDataFrame(normalized_products, schema=schema)

api_df.printSchema()
display(api_df)

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 16, Finished, Available, Finished, False)

root
 |-- id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- discountPercentage: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- stock: long (nullable = true)
 |-- availabilityStatus: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- weight: double (nullable = true)
 |-- dimensions: struct (nullable = true)
 |    |-- width: double (nullable = true)
 |    |-- height: double (nullable = true)
 |    |-- depth: double (nullable = true)
 |-- shippingInformation: string (nullable = true)
 |-- returnPolicy: string (nullable = true)
 |-- minimumOrderQuantity: long (nullable = true)
 |-- meta: struct (nullable = true)
 |    |-- createdAt: string (nullable = true)
 |    |-- updatedAt: string (nullable = true)
 |    |-- barcode: string (nullable = true)
 |    |-- qrCode: string (nullable = 

SynapseWidget(Synapse.DataFrame, 692ca38a-879b-4104-bb9e-8dad3b30bd73)

###### <mark>**Step 5: Write raw API data to Bronze**</mark>

In [15]:
# Write raw API response into Bronze Delta table
api_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_api_products_raw")

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 17, Finished, Available, Finished, False)

###### <mark>**Step 6: Validate Bronze**</mark>

In [16]:
# Validate raw Bronze table
spark.sql("""
SELECT *
FROM bronze_api_products_raw
""").show(truncate=False)

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 18, Finished, Available, Finished, False)

+---+-----------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+----------------+-------+------------------+------+-----+------------------+---------------+------+---------------------+--------------------------+---------------------+--------------------+-----------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------+
|id |title                                    |description                                                                                                                                                                              |category  |brand           |price  |discountPercentage|rating|stock|availabilityStatus|sku            |weight

###### <mark>**Step 7: Schema handling and flattening for Silver**</mark>

In [17]:
from pyspark.sql.functions import col

# Flatten selected nested fields and standardize schema
silver_api_products_df = api_df.select(
    col("id").cast("long").alias("product_id"),
    col("title").cast("string").alias("product_name"),
    col("description").cast("string").alias("description"),
    col("category").cast("string").alias("category"),
    col("brand").cast("string").alias("brand"),
    col("price").cast("double").alias("price"),
    col("discountPercentage").cast("double").alias("discount_percentage"),
    col("rating").cast("double").alias("rating"),
    col("stock").cast("long").alias("stock"),
    col("availabilityStatus").cast("string").alias("availability_status"),
    col("sku").cast("string").alias("sku"),
    col("weight").cast("double").alias("weight"),
    col("dimensions.width").cast("double").alias("dimension_width"),
    col("dimensions.height").cast("double").alias("dimension_height"),
    col("dimensions.depth").cast("double").alias("dimension_depth"),
    col("shippingInformation").cast("string").alias("shipping_information"),
    col("returnPolicy").cast("string").alias("return_policy"),
    col("minimumOrderQuantity").cast("long").alias("minimum_order_quantity"),
    col("meta.createdAt").cast("string").alias("meta_created_at"),
    col("meta.updatedAt").cast("string").alias("meta_updated_at"),
    col("thumbnail").cast("string").alias("thumbnail_url")
)

silver_api_products_df.printSchema()
display(silver_api_products_df)

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 19, Finished, Available, Finished, False)

root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- stock: long (nullable = true)
 |-- availability_status: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- weight: double (nullable = true)
 |-- dimension_width: double (nullable = true)
 |-- dimension_height: double (nullable = true)
 |-- dimension_depth: double (nullable = true)
 |-- shipping_information: string (nullable = true)
 |-- return_policy: string (nullable = true)
 |-- minimum_order_quantity: long (nullable = true)
 |-- meta_created_at: string (nullable = true)
 |-- meta_updated_at: string (nullable = true)
 |-- thumbnail_url: string (nullable = true)



SynapseWidget(Synapse.DataFrame, a5975e56-ef15-4998-905a-96b00a4df6c7)

###### <mark>**Step 8: Write Silver table**</mark>

In [18]:
# Write cleaned and flattened API data into Silver Delta table
silver_api_products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_api_products_clean")

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 20, Finished, Available, Finished, False)

###### <mark>**Step 9: Validate Silver table**</mark>

In [19]:
# Validate Silver output
spark.sql("""
SELECT *
FROM silver_api_products_clean
""").show(truncate=False)

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 21, Finished, Available, Finished, False)

+----------+-----------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+---------------+------+-------------------+------+-----+-------------------+---------------+------+---------------+----------------+---------------+--------------------------+---------------------+----------------------+------------------------+------------------------+--------------------------------------------------------------------------------------------+
|product_id|product_name                 |description                                                                                                                                                                              |category  |brand          |price |discount_percentage|rating|stock|availability_status|sku            |weight|dimension_width|dimension_height|dimension_depth|shipping_i

###### <mark>**A reusable function :- Retry fetch_api_with_retry()**</mark>

In [20]:
# RETRY FUNCTION
# Purpose:
# Safely call API with retries instead of failing on the first temporary error.

def fetch_api_with_retry(url, max_retries=3, sleep_seconds=2):
    """
    Calls the given API URL with retry logic.

    Parameters:
    - url: API endpoint to call
    - max_retries: number of retry attempts before failing
    - sleep_seconds: wait time between retries

    Returns:
    - response JSON as Python object if successful
    """

    attempt = 1

    while attempt <= max_retries:
        try:
            response = requests.get(url, timeout=30)

            # Success case
            if response.status_code == 200:
                print(f"Success on attempt {attempt}: {url}")
                return response.json()

            # Non-success HTTP response
            print(f"Attempt {attempt} failed with HTTP status {response.status_code}")

        except Exception as e:
            print(f"Attempt {attempt} failed with error: {e}")

        # Wait before retrying
        time.sleep(sleep_seconds)
        attempt += 1

    # If all retries fail, raise an exception
    raise Exception(f"API call failed after {max_retries} retries for URL: {url}")

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 22, Finished, Available, Finished, False)

###### <mark>**Pagination logic using limit and skip**</mark>

In [21]:
# PAGINATION INGESTION
# Purpose:
# Fetch all products from API in pages using limit + skip

all_products = []

limit = 10   # number of records per page
skip = 0     # offset
total_expected = None

while True:
    paged_url = f"https://dummyjson.com/products?limit={limit}&skip={skip}"

    # Call API using retry function
    page_data = fetch_api_with_retry(paged_url)

    # Capture total only once for reference
    if total_expected is None:
        total_expected = page_data["total"]
        print(f"Total records expected from API: {total_expected}")

    # Extract products from current page
    products = page_data["products"]

    # Stop when no more products
    if not products:
        print("No more products returned. Pagination complete.")
        break

    # Add current page products to overall list
    all_products.extend(products)

    print(f"Fetched page with skip={skip}, records={len(products)}")

    # Move to next page
    skip += limit

    # Stop if we already fetched expected total
    if len(all_products) >= total_expected:
        print("Fetched all expected records.")
        break

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 23, Finished, Available, Finished, False)

Success on attempt 1: https://dummyjson.com/products?limit=10&skip=0
Total records expected from API: 194
Fetched page with skip=0, records=10
Success on attempt 1: https://dummyjson.com/products?limit=10&skip=10
Fetched page with skip=10, records=10
Success on attempt 1: https://dummyjson.com/products?limit=10&skip=20
Fetched page with skip=20, records=10
Success on attempt 1: https://dummyjson.com/products?limit=10&skip=30
Fetched page with skip=30, records=10
Success on attempt 1: https://dummyjson.com/products?limit=10&skip=40
Fetched page with skip=40, records=10
Success on attempt 1: https://dummyjson.com/products?limit=10&skip=50
Fetched page with skip=50, records=10
Success on attempt 1: https://dummyjson.com/products?limit=10&skip=60
Fetched page with skip=60, records=10
Success on attempt 1: https://dummyjson.com/products?limit=10&skip=70
Fetched page with skip=70, records=10
Success on attempt 1: https://dummyjson.com/products?limit=10&skip=80
Fetched page with skip=80, reco

###### <mark>**Validate paginated API fetch**</mark>

In [22]:
# Validate total fetched records
print("Total products fetched through pagination:", len(all_products))

# Show first 2 paginated records
print(all_products[:2])

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 24, Finished, Available, Finished, False)

Total products fetched through pagination: 194
[{'id': 1, 'title': 'Essence Mascara Lash Princess', 'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.', 'category': 'beauty', 'price': 9.99, 'discountPercentage': 10.48, 'rating': 2.56, 'stock': 99, 'tags': ['beauty', 'mascara'], 'brand': 'Essence', 'sku': 'BEA-ESS-ESS-001', 'weight': 4, 'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99}, 'warrantyInformation': '1 week warranty', 'shippingInformation': 'Ships in 3-5 business days', 'availabilityStatus': 'In Stock', 'reviews': [{'rating': 3, 'comment': 'Would not recommend!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Eleanor Collins', 'reviewerEmail': 'eleanor.collins@x.dummyjson.com'}, {'rating': 4, 'comment': 'Very satisfied!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Lucas Gordon', 'reviewerEmail': 'lucas.gord

###### <mark>**Normalize paginated payload**</mark>

In [23]:
# Normalize paginated payload for strict schema compatibility

normalized_paginated_products = []

for product in all_products:
    product_copy = dict(product)

    # Convert top-level numeric fields to float
    if product_copy.get("price") is not None:
        product_copy["price"] = float(product_copy["price"])

    if product_copy.get("discountPercentage") is not None:
        product_copy["discountPercentage"] = float(product_copy["discountPercentage"])

    if product_copy.get("rating") is not None:
        product_copy["rating"] = float(product_copy["rating"])

    if product_copy.get("weight") is not None:
        product_copy["weight"] = float(product_copy["weight"])

    # Convert nested dimensions values to float
    if product_copy.get("dimensions") is not None:
        dims = dict(product_copy["dimensions"])

        if dims.get("width") is not None:
            dims["width"] = float(dims["width"])

        if dims.get("height") is not None:
            dims["height"] = float(dims["height"])

        if dims.get("depth") is not None:
            dims["depth"] = float(dims["depth"])

        product_copy["dimensions"] = dims

    normalized_paginated_products.append(product_copy)

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 25, Finished, Available, Finished, False)

###### <mark>**Create DataFrame from paginated fetch**</mark>

In [24]:
# Create Spark DataFrame using paginated and normalized API data

api_paginated_df = spark.createDataFrame(normalized_paginated_products, schema=schema)

api_paginated_df.printSchema()
display(api_paginated_df)

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 26, Finished, Available, Finished, False)

root
 |-- id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- discountPercentage: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- stock: long (nullable = true)
 |-- availabilityStatus: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- weight: double (nullable = true)
 |-- dimensions: struct (nullable = true)
 |    |-- width: double (nullable = true)
 |    |-- height: double (nullable = true)
 |    |-- depth: double (nullable = true)
 |-- shippingInformation: string (nullable = true)
 |-- returnPolicy: string (nullable = true)
 |-- minimumOrderQuantity: long (nullable = true)
 |-- meta: struct (nullable = true)
 |    |-- createdAt: string (nullable = true)
 |    |-- updatedAt: string (nullable = true)
 |    |-- barcode: string (nullable = true)
 |    |-- qrCode: string (nullable = 

SynapseWidget(Synapse.DataFrame, 6d8b5766-c7f1-43e6-af04-6a845ba4152b)

###### <mark>**write to separate Bronze/Silver pagination demo tables**</mark>

###### <mark>**Bronze pagination demo**</mark>

In [25]:
# Optional: write paginated raw API data to a separate Bronze demo table

api_paginated_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_api_products_paginated_raw")

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 27, Finished, Available, Finished, False)

###### <mark>**Flatten paginated Silver demo**</mark>

In [26]:
# Flatten paginated API data into Silver-style clean structure

silver_api_paginated_df = api_paginated_df.select(
    col("id").cast("long").alias("product_id"),
    col("title").cast("string").alias("product_name"),
    col("description").cast("string").alias("description"),
    col("category").cast("string").alias("category"),
    col("brand").cast("string").alias("brand"),
    col("price").cast("double").alias("price"),
    col("discountPercentage").cast("double").alias("discount_percentage"),
    col("rating").cast("double").alias("rating"),
    col("stock").cast("long").alias("stock"),
    col("availabilityStatus").cast("string").alias("availability_status"),
    col("sku").cast("string").alias("sku"),
    col("weight").cast("double").alias("weight"),
    col("dimensions.width").cast("double").alias("dimension_width"),
    col("dimensions.height").cast("double").alias("dimension_height"),
    col("dimensions.depth").cast("double").alias("dimension_depth"),
    col("shippingInformation").cast("string").alias("shipping_information"),
    col("returnPolicy").cast("string").alias("return_policy"),
    col("minimumOrderQuantity").cast("long").alias("minimum_order_quantity"),
    col("meta.createdAt").cast("string").alias("meta_created_at"),
    col("meta.updatedAt").cast("string").alias("meta_updated_at"),
    col("thumbnail").cast("string").alias("thumbnail_url")
)

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 28, Finished, Available, Finished, False)

###### <mark>**Write Silver pagination demo**</mark>

In [27]:
silver_api_paginated_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_api_products_paginated_clean")

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 29, Finished, Available, Finished, False)

###### <mark>**Validate final pagination output**</mark>

In [28]:
spark.sql("""
SELECT COUNT(*) AS total_products
FROM silver_api_products_paginated_clean
""").show()

spark.sql("""
SELECT *
FROM silver_api_products_paginated_clean
LIMIT 10
""").show(truncate=False)

StatementMeta(, 9cbb3bc2-ab2d-4d5e-8652-53aa44bc0f5e, 30, Finished, Available, Finished, False)

+--------------+
|total_products|
+--------------+
|           194|
+--------------+

+----------+-----------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------+-------+-------------------+------+-----+-------------------+---------------+------+---------------+----------------+---------------+--------------------------+---------------------+----------------------+------------------------+------------------------+-------------------------------------------------------------------------------------+
|product_id|product_name     |description                                                                                                                                                                                               |category   |brand |price  |discount_percentage|rating|stock|availability_status|sku    